# 사진 : 누락·분리·정션·추가 잉크 구조 점수

설문·추가 학습 없이 누락/분리/정션과 **기준으로 설명되지 않는 추가 잉크**를 감점합니다.
높을수록 측정 오류가 적습니다. 사람 선호나 글자 품질에 교정한 점수는 아닙니다.

1. 기본 csv 모드는 테스트 CSV에서 12장을 무작위로 골라 새로 평가합니다.
2. RANDOM_SEED를 바꾸면 다른 사진을 봅니다. INPUT_FILES에 실제 PNG 파일명 목록을 넣으면 그 사진만 평가합니다.
3. 첫 셀 → 모델 준비 → 점수 모듈 로드 → 계산 순서로 실행합니다.
3. 사진 아래의 **전체 점수**를 비교합니다. 카드의 **획별 점수 보기**를 누르면 세 오류와 획별 점수가 펼쳐집니다.

저장된 NPZ의 실제 평가 입력 배열을 우선 표시하고, 없으면 원본 PNG 경로에서 사진을 읽습니다.
기준 획 이미지나 진단 몽타주를 원본 대신 표시하지 않습니다. 경로가 없으면 이미지 없음으로 알립니다.
사진과 점수를 함께 담은 HTML/PNG도 결과 폴더에 저장합니다.

기존 구조 진단 노트북에 추가 셀을 붙이면 structure_results를 바로 재사용합니다.
기존 결과만 재계산하려면 MODE="auto"와 REPORT_PATH를 사용하세요.
csv 모드는 입력 사진, 목표 기준 이미지와 기준 획이 CSV에 있어야 하며 입력 획 정답 주석은 사용하지 않습니다.
MODE="values"는 숫자만 입력하는 테스트입니다. 기존 추출 모델을 재실행하지 않습니다.

기본 모드는 **획 추가·삭제를 강하게, 정션을 중간, 분리를 약하게** 감점하는 평가 규칙입니다.
분리 유지율 = 1 - 분리최대감점 × (wD·D/sD)/(1+wD·D/sD)
정션 유지율 = 1 - 정션최대감점 × (wJ·J/sJ)/(1+wJ·J/sJ)
획 점수 = 100 × (1-M)^wM × 분리 유지율 × 정션 유지율
평균/최악 획 혼합점수와 각 오류별 점수 상한 중 최솟값을 사용합니다.
삭제 상한 = 최악 획의 누락 유지율 × 100. 추가 상한 = 100 × exp(-wE·max(E-허용량,0)/추가scale).
단독 분리/정션은 기본값에서 최대 60/65점 감점이고, 복합 오류는 더 낮아질 수 있습니다.
카드의 '오류별 점수 상한 보기'에서 점수가 낮아진 원인을 확인하세요.

기본값에서 누락 1인 획은 0점, 세 오류가 0인 획은 100점입니다.
각 카드의 '추가 잉크 위치 보기'에서 빨간 영역을 확인하세요. 추가는 획 배정과 무관하게 입력 전체에서 측정합니다.
기준 위치는 제한된 범위에서 부드럽게 정렬합니다. 큰 변형이나 정렬 실패는 추가로 오검출할 수 있습니다.
반경과 허용량은 정상 자료에서 교정해야 합니다. 이름별로 점수나 오류 순서를 강제하지 않습니다.
오래된 JSON도 NPZ의 image/reference_geometry가 있으면 추가를 재측정합니다. 없으면 0으로 가정하지 않고 오류를 냅니다.


In [ ]:
# ===== 점수 설정: 누락·분리·정션 + 추가 잉크 =====
# auto: 현재 structure_results가 있으면 재사용, 없으면 REPORT_PATH 읽기
# csv: 테스트 CSV에서 다른 PNG 선택 → 획 추론 → 사진과 점수
# values: 아래 VALUES의 숫자만으로 바로 계산
MODE = "csv"
CHECKPOINT = "/content/drive/Shareddrives/2026 자율연구/HSQM/stroke_zip_runs/20261028_082616_066020/best.pt"
CSV_PATH = "/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/test_dataset/paths.csv"
INPUT_FILES = None  # 예: ["실제파일명.png", "다른파일명.png"]. None이면 무작위 선택
SAMPLE_COUNT = 12  # INPUT_FILES=None일 때 선택 장수. None = 전체 CSV
RANDOM_SEED = 42  # 숫자를 바꾸면 다른 표본. 같은 숫자는 같은 CSV 행 선택
DATASET_ROOT = None  # CSV 기준 경로로 이미지를 못 찾을 때 데이터 상위 폴더
STRUCTURE_CONFIG = {}  # 구조 측정 기본값 사용. 필요한 항목만 덮어쓰기
REPORT_PATH = "/content/drive/Shareddrives/2026 자율연구/HSQM/stroke_zip_runs/20261028_082616_066020/structure_20261101_092625_060835/structure_report.json"
SAVE_DIR = None  # None = 원래 결과 폴더 아래 indicator_시간. 직접 지정 시 새 폴더
SHOW_IMAGES = True  # 사진과 점수를 카드로 함께 표시
IMAGES_PER_ROW = 3
IMAGES_PER_PAGE = 12  # 저장용 PNG 한 장에 들어갈 사진 수
SHOW_STROKE_TABLE = False  # 기본은 카드의 '획별 점수 보기'를 눌러 확인
EXPORT_DEBUG = True  # 평가한 사진의 배열과 감점 내역만 진단 ZIP으로 저장

SCORE_CONFIG = dict(
    missing_weight=1.0,   # 누락 중요도. 0이면 이 항 제외
    split_weight=1.0,     # 분리 중요도
    junction_weight=1.0,  # 정션 중요도
    split_scale=0.2,      # 기본 모드: 이 크기에서 분리 최대 감점의 절반
    junction_scale=0.01,  # 기본 모드: 이 크기에서 정션 최대 감점의 절반
    worst_stroke_weight=0.5, # 글자 점수: 획 평균과 최악 획 점수의 혼합. 0=평균, 1=최악
    priority_mode=True,   # 삭제/추가 강한 감점, 정션/분리 제한 감점. False=이전 공식
    split_max_penalty=0.60, # 분리 단독 최대 감점 60점
    junction_max_penalty=0.65, # 정션 단독 최대 감점 65점
    extra_weight=1.0,     # 추가 잉크 감점. 0은 명시적으로 추가 검사를 끄는 설정
    extra_scale=0.025,    # 허용량 초과 2.5%p일 때 추가 점수 상한 100/e
    extra_allowance=0.03, # 정렬 후 남은 작은 잉크 차이 3%까지 감점 유예
    extra_radius=8.0,     # 목표 기준 중심선 주변의 허용 반경 (128 해상도 픽셀)
    extra_transition=0.5, # 허용 영역 경계의 부드러운 전이 폭
    extra_alignment=6,    # 기준 위치를 ±6px 안에서만 이동 허용
    extra_alignment_step=2, # 이동 후보 간격
    extra_alignment_temperature=0.01, # 부드러운 위치 선택 온도
    extra_alignment_prior=0.001, # 큰 이동을 억제하는 비용
    extra_alignment_sigma=2.0, # 위치 일치도를 계산하는 중심선 주변 폭
    eps=1e-6,
)

# MODE="values"일 때 사용. 이미지나 모델 없이 이 숫자만 계산합니다.
VALUES = [
    dict(stroke=1, missing=0.0, split=0.0, junction=0.0),
    dict(stroke=2, missing=0.809, split=0.0, junction=0.0),
]
EXTRA_VALUE = 0.0  # MODE="values": 글자 전체의 추가 잉크 비율을 직접 지정


In [ ]:
# CSV 모드에서만 모델 실행 환경 준비. auto/values는 다운로드하지 않습니다.
if MODE == "csv":
    #@title 2. 기존 V2 모델 코드 준비 (Colab에서 한 번 실행)
    from pathlib import Path
    import subprocess, sys, tempfile
    _structure_repo = Path(tempfile.mkdtemp(prefix="structure_runtime_", dir="/content"))
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/yechan25/hsqm.git", str(_structure_repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(_structure_repo / "requirements.txt")], check=True)
    sys.path.insert(0, str(_structure_repo))
    for _name in list(sys.modules):
        if _name == "stroke_model" or _name.startswith("stroke_model."):
            del sys.modules[_name]
    print("V2 준비 완료. 평가 모듈은 이 노트북에 포함된 버전을 사용합니다.")
    from google.colab import drive
    drive.mount("/content/drive")
    _indicator_sources = {'stroke_model.infer_reference': '"""V2 inference from CSV without input-stroke annotations or label conversion."""\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom stroke_model.convert_reference_csv import Resolver, existing_path, load_ink, path_list, normalized\nfrom stroke_model.reference_model import ReferenceStrokeModel\nfrom stroke_model.preview_reference import show_soft_strokes\n\n\ndef inference_columns(frame):\n    names = {str(c).lower(): c for c in frame.columns}\n    def find(*aliases):\n        for name in aliases:\n            if name.lower() in names:\n                return names[name.lower()]\n        raise ValueError(f\'Missing CSV column: {aliases}; found {list(frame.columns)}\')\n    return {\n        \'image\': find(\'I_prep\', \'input_image_path\', \'input_path\', \'input_image\', \'input\', \'source_path\'),\n        \'reference\': find(\'I_g\', \'reference_image\', \'reference_path\', \'target_image\', \'target_path\', \'glyph_path\'),\n        \'strokes\': find(\'I_G_strokes\', \'reference_strokes\', \'reference_stroke_paths\', \'template_strokes\', \'target_strokes\', \'target_stroke_paths\'),\n    }\n\n\ndef load_inference_row(row, columns, resolve, size):\n    # Deliberately never access input_stroke_paths or total_paths.\n    image_path = resolve(row[columns[\'image\']])\n    reference_path = resolve(row[columns[\'reference\']])\n    paths = path_list(row[columns[\'strokes\']])\n    if not paths:\n        raise ValueError(\'Reference stroke list is empty\')\n    image = torch.from_numpy(load_ink(image_path, size))\n    reference = torch.from_numpy(load_ink(reference_path, size))\n    strokes = torch.from_numpy(np.concatenate([load_ink(resolve(p), size) for p in paths]))\n    if not (strokes.flatten(1).max(1).values > .5).all():\n        raise ValueError(\'Empty reference stroke image\')\n    return image, reference, strokes, image_path, reference_path\n\n\ndef select_input_files(frame, image_column, input_files):\n    if isinstance(input_files, str):\n        input_files = [input_files]\n    values = [normalized(str(value).strip()) for value in frame[image_column]]\n    selected = []\n    for requested in input_files:\n        name = normalized(str(requested).strip())\n        # A bare filename matches a basename; a path matches the CSV path exactly.\n        matches = [i + 1 for i, value in enumerate(values)\n                   if (value == name if \'/\' in name else value.rsplit(\'/\', 1)[-1] == name)]\n        if not matches:\n            raise ValueError(f\'Input filename not found in CSV: {requested}\')\n        if len(matches) != 1:\n            raise ValueError(f\'Ambiguous input filename: {requested}; CSV rows {matches}. Specify the CSV input path or sample_rows.\')\n        if matches[0] not in selected:\n            selected.append(matches[0])\n    return selected\n\n\ndef run_inference_csv(checkpoint, csv_path, sample_rows=None, dataset_root=None,\n                      input_files=None,\n                      save_dir=None, show_soft=True):\n    """sample_rows: one-based CSV data row numbers; None processes all rows.\n\n    Outputs are stroke assignments, not calibrated missing/extra-stroke decisions.\n    No annotation metrics are computed. Existing checkpoints remain unchanged.\n    """\n    import matplotlib.pyplot as plt\n    import json\n\n    checkpoint, csv_path = existing_path(checkpoint), existing_path(csv_path)\n    frame = pd.read_csv(csv_path)\n    columns = inference_columns(frame)\n    if input_files is not None and sample_rows is not None:\n        raise ValueError(\'Specify input_files or sample_rows, not both\')\n    selected = (select_input_files(frame, columns[\'image\'], input_files) if input_files is not None\n                else list(range(1, len(frame) + 1)) if sample_rows is None else list(sample_rows))\n    if not selected or any(not isinstance(n, int) or not 1 <= n <= len(frame) for n in selected):\n        raise ValueError(f\'sample_rows must contain data row numbers in 1..{len(frame)}\')\n    resolve = Resolver(csv_path, existing_path(dataset_root) if dataset_root else csv_path.parent)\n    state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = ReferenceStrokeModel(image_size=state[\'image_size\'], width=state[\'width\'], pretrained=False)\n    model.load_state_dict(state[\'model\'], strict=True)\n    model = model.to(device).eval()\n    size = state[\'image_size\']\n    del state\n    destination = Path(save_dir) if save_dir else checkpoint.parent / (\'inference_\' + datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    destination.mkdir(parents=True, exist_ok=False)\n    results = []\n    print(f\'Inference: {len(selected)} samples, {device}. Input stroke annotations are not used.\')\n    for number in selected:\n        try:\n            image, reference, strokes, image_path, reference_path = load_inference_row(\n                frame.iloc[number - 1], columns, resolve, size)\n            with torch.inference_mode():\n                output = model(image[None].to(device), reference[None].to(device), strokes[None].to(device))\n            probabilities = output[\'probabilities\'][0].float().cpu().numpy()\n            masks = output[\'masks\'][0].float().cpu().numpy()\n            prediction = probabilities.argmax(0)\n            ink = image[0].numpy() > .5\n            labels = np.where(ink, prediction, -1)\n            prefix = destination / f\'row_{number:04d}\'\n            np.savez_compressed(str(prefix) + \'.npz\', probabilities=probabilities,\n                                soft_masks=masks, labels=labels, image=image.numpy(),\n                                reference=reference.numpy(), reference_strokes=strokes.numpy())\n            palette = plt.get_cmap(\'tab20\')(np.arange(len(strokes)) % 20)[:, :3]\n            colored = np.ones((*prediction.shape, 3))\n            colored[ink] = palette[prediction[ink]]\n            figure, axes = plt.subplots(1, 3, figsize=(10, 3))\n            for ax, (content, title) in zip(axes, [(image[0], \'Input\'), (reference[0], \'Reference\'), (colored, \'Predicted assignment\')]):\n                ax.imshow(content, cmap=\'gray_r\', vmin=0, vmax=1)\n                ax.set_title(title)\n                ax.axis(\'off\')\n            figure.suptitle(f\'CSV row {number}: {image_path.name} -> {reference_path.name}\')\n            figure.tight_layout()\n            figure.savefig(str(prefix) + \'.png\', dpi=140)\n            plt.show()\n            plt.close(figure)\n            if show_soft:\n                show_soft_strokes(image, strokes, output, prefix=prefix)\n            results.append({\'csv_row\': number, \'input\': str(image_path), \'reference\': str(reference_path),\n                            \'npz\': str(prefix) + \'.npz\', \'preview\': str(prefix) + \'.png\'})\n        except (ValueError, FileNotFoundError) as exc:\n            raise ValueError(f\'CSV data row {number}: {exc}\') from exc\n    report = {\'checkpoint\': str(checkpoint), \'csv\': str(csv_path), \'uses_input_stroke_annotations\': False,\n              \'results\': results}\n    (destination / \'inference.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Saved:\', destination)\n    return report\n', 'stroke_model.stroke_structure': '"""Reference-aware experimental stroke losses; inputs are absolute ink in [0, 1].\n\nReference preparation is discrete and cached. All prediction-dependent losses use\nPyTorch operations, including min/max subgradients. No predicted components are\ndiscarded. Scores are uncalibrated errors, not human-quality probabilities.\n"""\nfrom dataclasses import dataclass, fields\nimport math\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom scipy.ndimage import distance_transform_edt, label\nfrom skimage.morphology import skeletonize\n\n\n@dataclass(frozen=True)\nclass StructureConfig:\n    # All distances are pixels at the inference resolution (normally 128).\n    reference_threshold: float = 0.5\n    coverage_radius: int = 2\n    required_ink: float = 0.7\n    anchor_spacing: int = 10\n    anchor_radius: int = 1\n    corridor_radius: int = 3\n    pair_spans: tuple = (1, 2)\n    max_pairs: int = 32\n    connectivity_steps: int | None = None  # None: crop area, with convergence check\n    skeleton_iterations: int = 12\n    junction_radii: tuple = (1, 2, 3)\n    junction_margin: float = 0.0  # No font/short-branch exemption by default\n    junction_thresholds: tuple = (0.15, 0.30, 0.45, 0.60, 0.75)\n    junction_temperature: float = 0.025  # Soft occupancy transition, absolute ink\n    junction_arm_scale: float = 1.25\n    junction_arm_offset: float = 1.0\n    junction_min_arm_length: float = 2.0\n    junction_merge_radius: int = 1\n    junction_match_radius: int = 2\n    junction_min_levels: int = 2\n    junction_probe_scale: float = 1.5\n    junction_probe_offset: float = 1.0\n    junction_distance_steps: int = 12\n    junction_angular_samples: int = 48\n    junction_angular_smoothing: int = 3\n    junction_ring_margin: float = 0.1\n    junction_seed_radius: int = 2\n    attachment_steps: int | None = None  # None: H*W, with convergence check\n    expected_width: float = 5.0  # Optional normal-width prior, NOT skeleton width\n    width_radius: int = 4\n    width_tolerance: float = 0.25\n    missing_weight: float = 1.0\n    split_weight: float = 1.0\n    junction_weight: float = 1.0\n    width_weight: float = 0.0  # Enable only after normal-width calibration\n    eps: float = 1e-6\n\n    def __post_init__(self):\n        for name in (\'coverage_radius\', \'anchor_radius\', \'corridor_radius\',\n                     \'junction_seed_radius\', \'width_radius\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be a nonnegative integer\')\n        for name in (\'anchor_spacing\', \'max_pairs\', \'skeleton_iterations\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 1:\n                raise ValueError(f\'{name} must be a positive integer\')\n        if self.anchor_spacing <= 2 * self.anchor_radius + 1:\n            raise ValueError(\'anchor_spacing must exceed the anchor diameters\')\n        if not self.pair_spans or any(not isinstance(v, int) or v < 1 for v in self.pair_spans):\n            raise ValueError(\'pair_spans must contain positive integers\')\n        if not self.junction_radii or any(not isinstance(v, int) or v < 1 for v in self.junction_radii):\n            raise ValueError(\'junction_radii must contain integers >= 1\')\n        if (not self.junction_thresholds or any(not math.isfinite(float(v)) or not 0 < v < 1\n                                               for v in self.junction_thresholds)\n                or sorted(set(self.junction_thresholds)) != list(self.junction_thresholds)):\n            raise ValueError(\'junction_thresholds must be strictly increasing values in (0,1)\')\n        if not math.isfinite(self.junction_temperature) or self.junction_temperature <= 0:\n            raise ValueError(\'junction_temperature must be finite and positive\')\n        for name in (\'junction_arm_scale\', \'junction_min_arm_length\', \'junction_probe_scale\'):\n            if not math.isfinite(getattr(self, name)) or getattr(self, name) <= 0:\n                raise ValueError(f\'{name} must be positive\')\n        for name in (\'junction_arm_offset\', \'junction_probe_offset\', \'junction_ring_margin\'):\n            if not math.isfinite(getattr(self, name)) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be nonnegative\')\n        for name in (\'junction_merge_radius\', \'junction_match_radius\', \'junction_angular_smoothing\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be a nonnegative integer\')\n        if (not isinstance(self.junction_min_levels, int)\n                or not 1 <= self.junction_min_levels <= len(self.junction_thresholds)):\n            raise ValueError(\'junction_min_levels must fit the threshold count\')\n        if not isinstance(self.junction_distance_steps, int) or self.junction_distance_steps < 1:\n            raise ValueError(\'junction_distance_steps must be a positive integer\')\n        if not isinstance(self.junction_angular_samples, int) or self.junction_angular_samples < 16:\n            raise ValueError(\'junction_angular_samples must be an integer >= 16\')\n        for name in (\'connectivity_steps\', \'attachment_steps\'):\n            v = getattr(self, name)\n            if v is not None and (not isinstance(v, int) or v < 1):\n                raise ValueError(f\'{name} must be None or a positive integer\')\n        if not 0 < self.required_ink <= 1 or not 0 < self.reference_threshold < 1:\n            raise ValueError(\'required_ink must be in (0,1]; reference_threshold in (0,1)\')\n        if self.expected_width <= 0 or self.eps <= 0 or not 0 <= self.width_tolerance < 1:\n            raise ValueError(\'Invalid width or eps\')\n        if self.junction_margin < 0 or any(getattr(self, n) < 0 for n in\n                (\'missing_weight\', \'split_weight\', \'junction_weight\', \'width_weight\')):\n            raise ValueError(\'Weights and junction_margin must be nonnegative\')\n        if any(not math.isfinite(float(getattr(self, f.name))) for f in fields(self)\n               if isinstance(getattr(self, f.name), (float, int))):\n            raise ValueError(\'Configuration values must be finite\')\n\n\ndef _pool(x, radius, mean=False):\n    k = 2 * radius + 1\n    return (F.avg_pool2d(x, k, 1, radius) if mean else F.max_pool2d(x, k, 1, radius))\n\n\ndef soft_skeleton(x, iterations):\n    """Pooling skeleton from soft-clDice; a surrogate, not exact graph thinning."""\n    def erode(v):\n        return torch.minimum(-F.max_pool2d(-v, (3, 1), 1, (1, 0)),\n                             -F.max_pool2d(-v, (1, 3), 1, (0, 1)))\n    def opening(v):\n        return F.max_pool2d(erode(v), 3, 1, 1)\n    skel = F.relu(x - opening(x))\n    for _ in range(iterations):\n        x = erode(x)\n        delta = F.relu(x - opening(x))\n        skel = skel + F.relu(delta - skel * delta)\n    return skel\n\n\ndef reconstruct(mask, seed, steps=None):\n    """8-connected grayscale reconstruction; no dilation/closing of mask.\n\n    Returns (reachable strength, converged). None allows H*W iterations, a\n    conservative finite-grid bound. Early stopping reads detached values only;\n    the returned tensor retains the graph. Explicit small step caps can truncate.\n    """\n    reached = torch.minimum(mask, seed)\n    limit = mask.shape[-2] * mask.shape[-1] if steps is None else steps\n    converged = False\n    for _ in range(limit):\n        updated = torch.minimum(mask, F.max_pool2d(reached, 3, 1, 1))\n        if torch.equal(updated.detach(), reached.detach()):\n            reached = updated\n            converged = True\n            break\n        reached = updated\n    return reached, converged\n\n\ndef _ordered_centerline(binary):\n    skel = skeletonize(binary)\n    _, count = label(skel, structure=np.ones((3, 3)))\n    if count != 1:\n        raise ValueError(\'Reference stroke must have one connected centerline; clean the reference\')\n    points = [tuple(v) for v in np.argwhere(skel)]\n    lookup = set(points)\n    neighbors = {}\n    for y, x in points:\n        adjacent = []\n        for dy in (-1, 0, 1):\n            for dx in (-1, 0, 1):\n                q = (y + dy, x + dx)\n                if (not dy and not dx) or q not in lookup:\n                    continue\n                # Avoid triangular shortcut edges at a normal digital corner.\n                if dy and dx and ((y + dy, x) in lookup or (y, x + dx) in lookup):\n                    continue\n                adjacent.append(q)\n        neighbors[(y, x)] = adjacent\n    if any(len(v) > 2 for v in neighbors.values()):\n        raise ValueError(\'Reference skeleton branches; provide a clean unbranched reference stroke\')\n    endpoints = sorted(p for p, n in neighbors.items() if len(n) == 1)\n    closed = len(points) > 2 and not endpoints\n    if len(points) < 5 or (not closed and len(endpoints) != 2):\n        raise ValueError(\'Reference centerline must be a nontrivial open curve or closed ring\')\n    start = endpoints[0] if endpoints else min(points)\n    order, visited, current = [], set(), start\n    while current not in visited:\n        order.append(current)\n        visited.add(current)\n        choices = sorted(q for q in neighbors[current] if q not in visited)\n        if not choices:\n            break\n        current = choices[0]\n    if len(order) != len(points):\n        raise ValueError(\'Cannot order reference centerline without losing pixels\')\n    return skel, np.asarray(order), closed\n\n\ndef _disk_region(shape, point, radius):\n    yy, xx = np.indices(shape)\n    return ((yy - point[0]) ** 2 + (xx - point[1]) ** 2 <= radius ** 2)\n\n\nclass _Pair(nn.Module):\n    def __init__(self, corridor, a, b, slices):\n        super().__init__()\n        for name, value in [(\'corridor\', corridor), (\'a\', a), (\'b\', b)]:\n            self.register_buffer(name, torch.tensor(value, dtype=torch.float32)[None, None])\n        self.slices = slices\n\n\nclass _Stroke(nn.Module):\n    def __init__(self, reference, config):\n        super().__init__()\n        skel, order, self.closed = _ordered_centerline(reference > config.reference_threshold)\n        self.length = len(order)\n        self.register_buffer(\'skeleton\', torch.tensor(skel, dtype=torch.float32)[None, None])\n        self.register_buffer(\'ys\', torch.tensor(order[:, 0], dtype=torch.long))\n        self.register_buffer(\'xs\', torch.tensor(order[:, 1], dtype=torch.long))\n        distance = distance_transform_edt(~skel)\n        self.register_buffer(\'seeds\', torch.tensor(distance <= config.junction_seed_radius,\n                                                   dtype=torch.float32)[None, None])\n        normal = distance <= max(0.5, config.expected_width / 2)\n        self.register_buffer(\'normal\', torch.tensor(normal, dtype=torch.float32)[None, None])\n        anchors = list(range(0, len(order), config.anchor_spacing))\n        if not self.closed and anchors[-1] != len(order) - 1:\n            anchors.append(len(order) - 1)\n        raw_pairs = []\n        for span in config.pair_spans:\n            for i in range(len(anchors)):\n                j = i + span\n                if not self.closed and j >= len(anchors):\n                    continue\n                if self.closed and span >= len(anchors):\n                    continue\n                begin, end = anchors[i], anchors[j % len(anchors)]\n                path = (order[begin:end + 1] if end > begin else\n                        np.concatenate((order[begin:], order[:end + 1])))\n                if len(path) <= 2 * config.anchor_radius + 2:\n                    continue\n                # Keep anchor patches disjoint, including on tight curves.\n                if np.linalg.norm(order[begin] - order[end]) <= 2 * config.anchor_radius + 1:\n                    continue\n                segment = np.zeros_like(skel)\n                segment[path[:, 0], path[:, 1]] = True\n                corridor = distance_transform_edt(~segment) <= config.corridor_radius\n                a = _disk_region(skel.shape, order[begin], config.anchor_radius) & corridor\n                b = _disk_region(skel.shape, order[end], config.anchor_radius) & corridor\n                yy, xx = np.where(corridor)\n                slices = (slice(yy.min(), yy.max() + 1), slice(xx.min(), xx.max() + 1))\n                raw_pairs.append((corridor[slices], a[slices], b[slices], slices))\n        if len(raw_pairs) > config.max_pairs:\n            ids = np.linspace(0, len(raw_pairs) - 1, config.max_pairs).round().astype(int)\n            raw_pairs = [raw_pairs[i] for i in ids]\n        self.pairs = nn.ModuleList([_Pair(*args) for args in raw_pairs])\n        if not self.pairs:\n            raise ValueError(\'No valid anchor pairs: reduce anchor_spacing/radius for this reference\')\n\n\ndef _ring_offsets(radius):\n    # Sample a digital square perimeter in cyclic order, with no duplicates.\n    return ([( -radius, x) for x in range(-radius, radius)] +\n            [(y, radius) for y in range(-radius, radius)] +\n            [(radius, x) for x in range(radius, -radius, -1)] +\n            [(y, -radius) for y in range(radius, -radius, -1)])\n\n\ndef _crossings(skeleton, radius):\n    h, w = skeleton.shape[-2:]\n    padded = F.pad(skeleton, (radius,) * 4)\n    samples = [padded[..., radius + y:radius + y + h, radius + x:radius + x + w]\n               for y, x in _ring_offsets(radius)]\n    return sum((a - b).abs() for a, b in zip(samples, samples[1:] + samples[:1])) / 2\n\n\ndef soft_occupancy(mask, threshold, temperature):\n    """Approximate absolute level-set occupancy, with exactly 0->0 and 1->1.\n\n    This is NOT per-stroke max normalization. An empty/faint missing stroke is\n    never amplified relative to its own maximum. Shape and absolute ink are\n    separated using shared, configurable thresholds.\n    """\n    tau = torch.as_tensor(threshold, device=mask.device, dtype=mask.dtype)\n    low = torch.sigmoid(-tau / temperature)\n    high = torch.sigmoid((1 - tau) / temperature)\n    return ((torch.sigmoid((mask - tau) / temperature) - low) / (high - low)).clamp(0, 1)\n\n\ndef soft_distance_to_background(occupancy, steps):\n    """Differentiable truncated chamfer distance, in input pixels.\n\n    Fixed iterations; min has the same subgradient qualification as pooling.\n    This estimates the probe scale; it never edits or fills the input mask.\n    """\n    distance = occupancy * steps\n    h, w = occupancy.shape[-2:]\n    for _ in range(steps):\n        padded = F.pad(distance, (1, 1, 1, 1), value=0)\n        neighbors = [distance]\n        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)):\n            neighbors.append(padded[..., 1+dy:1+dy+h, 1+dx:1+dx+w] + math.hypot(dy, dx))\n        distance = torch.stack(neighbors).amin(0)\n    return distance\n\n\ndef occupancy_ring_arms(occupancy, radius, samples, smoothing):\n    """Count resolved angular transitions on the actual occupancy, not skeleton."""\n    n, _, h, w = occupancy.shape\n    angles = torch.arange(samples, device=occupancy.device, dtype=occupancy.dtype) * (2*math.pi/samples)\n    yy, xx = torch.meshgrid(torch.arange(h, device=occupancy.device, dtype=occupancy.dtype),\n                            torch.arange(w, device=occupancy.device, dtype=occupancy.dtype), indexing=\'ij\')\n    x = xx[None, :, :, None] + radius[:, 0, :, :, None] * angles.cos()\n    y = yy[None, :, :, None] + radius[:, 0, :, :, None] * angles.sin()\n    grid = torch.stack(((x+.5)*2/w-1, (y+.5)*2/h-1), -1).reshape(n, h, w*samples, 2)\n    ring = F.grid_sample(occupancy, grid, align_corners=False).reshape(n, 1, h, w, samples)\n    for _ in range(smoothing):\n        ring = (ring.roll(1, -1) + 2*ring + ring.roll(-1, -1))/4\n    return (ring-ring.roll(1, -1)).abs().sum(-1)/2\n\n\ndef multilevel_junction(mask, seeds, config):\n    """Differentiate shape per absolute level BEFORE counting skeleton arms.\n\n    Levels are stacked on the batch axis. Return a strongest-level response map\n    and level diagnostics. A branch with ink .6 is not counted as 3*.6 arms.\n    """\n    b, _, h, w = mask.shape\n    levels = len(config.junction_thresholds)\n    tau = mask.new_tensor(config.junction_thresholds)[:, None, None, None, None]\n    occupancy = soft_occupancy(mask[None], tau, config.junction_temperature).reshape(levels*b, 1, h, w)\n    skeleton = soft_skeleton(occupancy, config.skeleton_iterations)\n    attached, converged = reconstruct(occupancy, occupancy * seeds, config.attachment_steps)\n    local = torch.stack([F.relu(_crossings(skeleton, r)-2-config.junction_margin).square()\n                         for r in config.junction_radii]).mean(0)\n    local = F.max_pool2d(local, 5, 1, 2)\n    distance = soft_distance_to_background(occupancy, config.junction_distance_steps)\n    radius = config.junction_probe_scale*distance + config.junction_probe_offset\n    arms = occupancy_ring_arms(occupancy, radius, config.junction_angular_samples,\n                               config.junction_angular_smoothing)\n    resolved = F.relu(arms-2-config.junction_ring_margin).square()\n    maps = (resolved * local * skeleton * attached).reshape(levels, b, 1, h, w)\n    # Average evidence instead of selecting a worst pixel from any one threshold.\n    response = maps.mean(0)\n    return response, maps, converged\n\n\nclass StrokeStructureLoss(nn.Module):\n    """Prepare once on fixed aligned reference strokes, then call on live masks.\n\n    reference_strokes: K,H,W. forward: K,H,W or B,K,H,W, floating [0,1].\n    Returned per_stroke values are B,K. loss is an EXTRACTOR structural loss,\n    not a source-image quality score. Reference deficits are returned separately.\n    Closed curves are ordered\n    cyclically; local corridors prevent bypassing a gap via the rest of a ring.\n    No alignment is estimated here. Geometry must already be in input coordinates.\n    """\n    def __init__(self, reference_strokes, config=None):\n        super().__init__()\n        self.config = config or StructureConfig()\n        ref = torch.as_tensor(reference_strokes).detach().float().cpu().numpy()\n        if ref.ndim != 3 or not np.isfinite(ref).all() or (ref < 0).any() or (ref > 1).any():\n            raise ValueError(\'reference_strokes must be finite K,H,W in [0,1]\')\n        self.shape = ref.shape\n        prepared = []\n        for k, stroke in enumerate(ref):\n            try:\n                prepared.append(_Stroke(stroke, self.config))\n            except ValueError as exc:\n                raise ValueError(f\'Reference stroke {k + 1}: {exc}\') from exc\n        if not prepared:\n            raise ValueError(\'At least one reference stroke is required\')\n        self.strokes = nn.ModuleList(prepared)\n\n    def forward(self, masks, return_maps=False):\n        if masks.ndim == 3:\n            masks = masks[None]\n        if masks.ndim != 4 or tuple(masks.shape[1:]) != tuple(self.shape):\n            raise ValueError(f\'masks must have trailing shape {self.shape}\')\n        if not masks.is_floating_point():\n            raise ValueError(\'masks must be floating point\')\n        if not bool(torch.isfinite(masks).all()) or bool((masks < 0).any()) or bool((masks > 1).any()):\n            raise ValueError(\'masks must be finite in [0,1]; do not normalize each stroke\')\n        if next(self.buffers()).device != masks.device:\n            raise ValueError(\'Move the prepared loss to masks.device with .to(device)\')\n        cfg = self.config\n        metrics = {name: [] for name in (\'missing\', \'split\', \'junction\', \'width\')}\n        all_maps, convergence = [], []\n        for k, stroke in enumerate(self.strokes):\n            m = masks[:, k:k + 1].float()\n            coverage = _pool(m, cfg.coverage_radius)[:, 0, stroke.ys, stroke.xs]\n            deficits = F.relu(1 - coverage / cfg.required_ink).square()\n            missing = deficits.mean(-1)\n            local_mass = _pool(m, cfg.width_radius, mean=True)[:, 0, stroke.ys, stroke.xs]\n            expected = _pool(stroke.normal, cfg.width_radius, mean=True)[:, 0, stroke.ys, stroke.xs]\n            width = F.relu(1 - local_mass / (expected * (1 - cfg.width_tolerance) + cfg.eps)).square().mean(-1)\n            split_values, split_maps, pair_converged = [], [], []\n            for pair in stroke.pairs:\n                f = m[..., pair.slices[0], pair.slices[1]] * pair.corridor\n                reached, converged = reconstruct(f, f * pair.a, cfg.connectivity_steps)\n                u = (f * pair.a).flatten(1).amax(1)\n                v = (f * pair.b).flatten(1).amax(1)\n                c = (reached * pair.b).flatten(1).amax(1)\n                strength = torch.minimum(u, v)\n                # clamp avoids the tiny nonzero epsilon penalty on perfect paths.\n                gap = F.relu((strength - c) / strength.clamp_min(cfg.eps)).square()\n                split_values.append(u * v * gap)\n                pair_converged.append(converged)\n                if return_maps:\n                    split_maps.append((pair, F.relu(f - reached)))\n            split = torch.stack(split_values, -1).mean(-1)\n            attached, attach_converged = reconstruct(m, m * stroke.seeds, cfg.attachment_steps)\n            junction_map, junction_levels, junction_converged = multilevel_junction(m, stroke.seeds, cfg)\n            junction = junction_map.flatten(1).sum(-1) / stroke.length\n            for name, value in [(\'missing\', missing), (\'split\', split), (\'junction\', junction), (\'width\', width)]:\n                metrics[name].append(value)\n            convergence.append({\'stroke\': k + 1, \'pairs\': all(pair_converged),\n                                \'attachment\': attach_converged and junction_converged, \'closed\': stroke.closed})\n            if return_maps:\n                missing_map = torch.zeros_like(m)\n                missing_map[:, 0, stroke.ys, stroke.xs] = deficits\n                split_map = torch.zeros_like(m)\n                for pair, area in split_maps:\n                    # Maps are diagnostics, not the differentiable loss path.\n                    with torch.no_grad():\n                        target = split_map[..., pair.slices[0], pair.slices[1]]\n                        target.copy_(torch.maximum(target, area.detach()))\n                all_maps.append({\'mask\': m.detach(), \'centerline\': stroke.skeleton.detach(),\n                                 \'missing\': missing_map.detach(), \'split\': split_map.detach(),\n                                 \'junction\': junction_map.detach(), \'attached\': attached.detach(),\n                                 \'junction_level_scores\': (junction_levels.flatten(2).sum(-1) / stroke.length).detach()})\n        per_stroke = {key: torch.stack(value, -1) for key, value in metrics.items()}\n        extraction = cfg.split_weight * per_stroke[\'split\'] + cfg.junction_weight * per_stroke[\'junction\']\n        reference = cfg.missing_weight * per_stroke[\'missing\'] + cfg.width_weight * per_stroke[\'width\']\n        result = {\'loss\': extraction.mean(), \'reference_loss\': reference.mean(),\n                  \'per_stroke\': {**per_stroke, \'extraction_error\': extraction,\n                                 \'reference_deficit\': reference},\n                  \'source_quality\': {\'assessed\': False,\n                      \'reason\': \'Extracted-mask defects do not identify source-image defects.\'},\n                  \'convergence\': convergence}\n        if return_maps:\n            result[\'maps\'] = all_maps\n        return result\n', 'stroke_model.junction_diagnostics': '"""Non-differentiable branch verification on absolute level-set skeletons.\n\nThis independent diagnostic must never be added to a training/guidance loss.\nIt checks the differentiable surrogate against actual thinned binary masks.\n"""\nimport heapq\nimport math\nimport numpy as np\nfrom scipy.ndimage import label, center_of_mass, distance_transform_edt, binary_dilation\nfrom skimage.morphology import skeletonize\n\ndef skeleton_graph(skeleton):\n    """8-neighbor graph, omitting diagonal shortcuts around occupied corners."""\n    pixels = set(map(tuple, np.argwhere(skeleton)))\n    graph = {}\n    for y, x in pixels:\n        neighbors = []\n        for dy in (-1, 0, 1):\n            for dx in (-1, 0, 1):\n                q = (y+dy, x+dx)\n                if (not dy and not dx) or q not in pixels:\n                    continue\n                if dy and dx and ((y+dy, x) in pixels or (y, x+dx) in pixels):\n                    continue\n                neighbors.append(q)\n        graph[y, x] = neighbors\n    return graph\n\n\ndef resolved_arms(graph, root, radius):\n    """Count independent arms that leave a local junction core, not pixel tips.\n\n    Paths joining each other inside the core count once. Endpoints inside the\n    core are unresolved; this does NOT establish that a short feature is normal.\n    """\n    distances = {root: 0.}\n    queue = [(0., root)]\n    exits = set()\n    while queue:\n        distance, p = heapq.heappop(queue)\n        if distance != distances[p]:\n            continue\n        for q in graph[p]:\n            step = math.hypot(q[0]-p[0], q[1]-p[1])\n            value = distance + step\n            if distances.get(q, math.inf) <= value:\n                continue\n            if value > radius:\n                exits.add(p)\n            elif value < distances.get(q, math.inf):\n                distances[q] = value\n                heapq.heappush(queue, (value, q))\n    remaining = set(distances) - {root}\n    count = 0\n    while remaining:\n        start = remaining.pop()\n        component, stack = {start}, [start]\n        while stack:\n            p = stack.pop()\n            for q in graph[p]:\n                if q in remaining:\n                    remaining.remove(q)\n                    component.add(q)\n                    stack.append(q)\n        reaches = bool(component & exits) or any(distances[p] >= radius-1e-6 for p in component)\n        if reaches and any(p in component for p in graph[root]):\n            count += 1\n    return count\n\n\ndef _regions(candidates, merge_radius):\n    if not candidates.any():\n        return 0, []\n    if merge_radius:\n        yy, xx = np.mgrid[-merge_radius:merge_radius+1, -merge_radius:merge_radius+1]\n        expanded = binary_dilation(candidates, (xx*xx+yy*yy) <= merge_radius**2)\n    else:\n        expanded = candidates\n    regions, count = label(expanded, structure=np.ones((3, 3)))\n    centers = [tuple(float(v) for v in point) for point in\n               center_of_mass(candidates, regions, range(1, count+1))]\n    return int(count), centers\n\n\ndef diagnose_junction(mask, seed_region, thresholds, *, arm_scale=1.25,\n                      arm_offset=1., min_arm_length=2., merge_radius=1,\n                      match_radius=2, min_levels=2):\n    ink = np.asarray(mask, dtype=np.float32)\n    seed = np.asarray(seed_region, dtype=bool)\n    if ink.ndim != 2 or seed.shape != ink.shape:\n        raise ValueError(\'mask and seed_region must have matching H,W shapes\')\n    levels, artifacts = [], []\n    for threshold in thresholds:\n        foreground = ink >= threshold\n        components, _ = label(foreground, structure=np.ones((3, 3)))\n        marked = np.unique(components[foreground & seed])\n        marked = marked[marked > 0]\n        attached = np.isin(components, marked) if len(marked) else np.zeros_like(foreground)\n        skeleton = skeletonize(attached)\n        graph = skeleton_graph(skeleton)\n        raw = np.zeros_like(skeleton)\n        branches = np.zeros_like(skeleton)\n        radius_field = distance_transform_edt(attached)\n        for p in graph:\n            if len(graph[p]) >= 3:\n                raw[p] = True\n                radius = max(min_arm_length, arm_scale*radius_field[p] + arm_offset)\n                if resolved_arms(graph, p, radius) >= 3:\n                    branches[p] = True\n        count, centers = _regions(branches, merge_radius)\n        uncertain = raw & ~branches\n        uncertain_count, uncertain_centers = _regions(uncertain, merge_radius)\n        levels.append({\'threshold\': float(threshold), \'branch_regions\': int(count),\n                       \'unresolved_regions\': uncertain_count,\n                       \'branch_pixels\': int(branches.sum()), \'skeleton_pixels\': int(skeleton.sum())})\n        artifacts.append({\'skeleton\': skeleton, \'branches\': branches, \'raw\': raw, \'centers\': centers,\n                          \'uncertain\': uncertain, \'uncertain_centers\': uncertain_centers})\n    yy, xx = np.mgrid[-match_radius:match_radius+1, -match_radius:match_radius+1]\n    footprint = (xx*xx+yy*yy) <= match_radius**2\n    votes = sum(binary_dilation(a[\'branches\'], footprint).astype(np.int16) for a in artifacts)\n    stable_region = votes >= min_levels\n    candidate_max = max(item[\'branch_regions\'] for item in levels)\n    for item, artifact in zip(levels, artifacts):\n        artifact[\'branches\'] = artifact[\'branches\'] & stable_region\n        artifact[\'uncertain\'] = artifact[\'raw\'] & ~artifact[\'branches\']\n        count, centers = _regions(artifact[\'branches\'], merge_radius)\n        uncertain_count, uncertain_centers = _regions(artifact[\'uncertain\'], merge_radius)\n        item[\'branch_regions\'] = count\n        item[\'unresolved_regions\'] = uncertain_count\n        item[\'branch_pixels\'] = int(artifact[\'branches\'].sum())\n        artifact[\'centers\'], artifact[\'uncertain_centers\'] = centers, uncertain_centers\n    # Select the clearest branch-containing level; preserve the full level table.\n    selected = max(range(len(levels)), key=lambda i: (levels[i][\'branch_regions\'],\n                                                     levels[i][\'skeleton_pixels\']))\n    hits = sum(item[\'branch_regions\'] > 0 for item in levels)\n    return {\'branch_regions_max\': max(item[\'branch_regions\'] for item in levels),\n            \'candidate_regions_max\': candidate_max,\n            \'unresolved_regions_max\': max(item[\'unresolved_regions\'] for item in levels),\n            \'levels_with_branches\': hits, \'tested_levels\': len(levels),\n            \'selected_threshold\': levels[selected][\'threshold\'], \'levels\': levels,\n            **artifacts[selected]}\n', 'stroke_model.score_stroke_structure': '"""CSV/NPZ evaluation and plots for the experimental structure losses."""\nfrom dataclasses import asdict\nfrom datetime import datetime\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom stroke_model.stroke_structure import StructureConfig, StrokeStructureLoss\nfrom stroke_model.junction_diagnostics import diagnose_junction\n\n\ndef _destination(save_dir):\n    destination = Path(save_dir) if save_dir else Path(\'structure_results\') / datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\')\n    destination.mkdir(parents=True, exist_ok=False)\n    return destination\n\n\ndef _measure(masks, geometry, config, prefix, reference=None, image=None,\n             show=True, caption=\'\', show_details=False, target_image=None):\n    evaluator = StrokeStructureLoss(geometry, config).to(masks.device)\n    with torch.no_grad():\n        result = evaluator(masks, return_maps=True)\n    rows, diagnostics = [], []\n    for k in range(len(geometry)):\n        diagnostic = diagnose_junction(masks[k].detach().float().cpu().numpy(),\n            evaluator.strokes[k].seeds[0, 0].cpu().numpy(), config.junction_thresholds,\n            arm_scale=config.junction_arm_scale, arm_offset=config.junction_arm_offset,\n            min_arm_length=config.junction_min_arm_length, merge_radius=config.junction_merge_radius,\n            match_radius=config.junction_match_radius, min_levels=config.junction_min_levels)\n        diagnostics.append(diagnostic)\n        rows.append({\'stroke\': k + 1,\n                     **{key: float(value[0, k]) for key, value in result[\'per_stroke\'].items()},\n                     \'branch_regions_max\': diagnostic[\'branch_regions_max\'],\n                     \'levels_with_branches\': diagnostic[\'levels_with_branches\'],\n                     \'unresolved_regions_max\': diagnostic[\'unresolved_regions_max\'],\n                     \'junction_peak_response\': float(result[\'maps\'][k][\'junction\'].max()),\n                     \'closed_reference\': result[\'convergence\'][k][\'closed\'],\n                     \'connectivity_converged\': result[\'convergence\'][k][\'pairs\'],\n                     \'attachment_converged\': result[\'convergence\'][k][\'attachment\']})\n    frame = pd.DataFrame(rows)\n    frame.to_csv(str(prefix) + \'_errors.csv\', index=False, encoding=\'utf-8-sig\')\n    payload = {\'soft_masks\': masks.detach().float().cpu().numpy(),\n               \'reference_geometry\': torch.as_tensor(geometry).detach().cpu().numpy()}\n    if reference is not None:\n        payload[\'reference_strokes\'] = torch.as_tensor(reference).detach().cpu().numpy()\n    if image is not None:\n        payload[\'image\'] = torch.as_tensor(image).detach().cpu().numpy()\n    if target_image is not None:\n        payload[\'reference_image\'] = torch.as_tensor(target_image).detach().cpu().numpy()\n    for name in (\'centerline\', \'missing\', \'split\', \'junction\', \'attached\'):\n        payload[name + \'_maps\'] = np.stack([m[name][0, 0].cpu().numpy() for m in result[\'maps\']])\n    payload[\'junction_thresholds\'] = np.asarray(config.junction_thresholds)\n    payload[\'junction_level_scores\'] = np.stack([m[\'junction_level_scores\'][:, 0].cpu().numpy() for m in result[\'maps\']])\n    payload[\'junction_hard_skeletons\'] = np.stack([d[\'skeleton\'] for d in diagnostics])\n    payload[\'junction_hard_branches\'] = np.stack([d[\'branches\'] for d in diagnostics])\n    payload[\'junction_unresolved\'] = np.stack([d[\'uncertain\'] for d in diagnostics])\n    np.savez_compressed(str(prefix) + \'_structure.npz\', **payload)\n    figures = _plot_junctions(result[\'maps\'], rows, diagnostics, prefix, caption, show)\n    figures += _plot(result[\'maps\'], rows, prefix, caption, show and show_details)\n    print(caption)\n    columns = [\'stroke\', \'branch_regions_max\', \'unresolved_regions_max\', \'levels_with_branches\', \'junction\', \'split\', \'missing\']\n    print(frame[columns].rename(columns={\'junction\': \'J_soft\'}).to_string(index=False,\n        formatters={key: lambda x: f\'{x:.3g}\' for key in (\'J_soft\', \'split\', \'missing\')}))\n    print(\'Red = resolved/persistent branch; amber = unresolved candidate. J_soft remains an experimental surrogate.\')\n    if not frame[\'connectivity_converged\'].all() or not frame[\'attachment_converged\'].all():\n        print(\'Iteration cap reached: increase connectivity_steps/attachment_steps before interpreting these errors.\')\n    return {\'strokes\': rows, \'mean_errors\': {key: float(value.mean()) for key, value in result[\'per_stroke\'].items()},\n            \'worst_extraction_error\': float(result[\'per_stroke\'][\'extraction_error\'].max()),\n            \'source_quality\': result[\'source_quality\'],\n            \'junction_checks\': [{key: value for key, value in d.items()\n                                 if key not in (\'skeleton\', \'branches\', \'raw\', \'uncertain\')} for d in diagnostics],\n            \'csv\': str(prefix) + \'_errors.csv\', \'npz\': str(prefix) + \'_structure.npz\', \'figures\': figures}\n\n\ndef _plot_junctions(maps, rows, diagnostics, prefix, caption, show):\n    """Three rows: original mask / independently thinned graph / soft response."""\n    import matplotlib.pyplot as plt\n    import unicodedata\n    paths = []\n    for offset in range(0, len(maps), 4):\n        page = maps[offset:offset+4]\n        fig, axes = plt.subplots(3, len(page), squeeze=False, figsize=(3.8*len(page), 10), layout=\'constrained\')\n        page_max = max(float(item[\'junction\'].max()) for item in page)\n        display_max = max(page_max, 1e-8)\n        for col, item in enumerate(page):\n            k = offset + col\n            diagnostic, values = diagnostics[k], rows[k]\n            ink = item[\'mask\'][0, 0].cpu().numpy()\n            axes[0, col].imshow(ink, cmap=\'gray_r\', vmin=0, vmax=1)\n            axes[0, col].set_title(f\'Stroke {k+1}: original soft mask\')\n            axes[1, col].imshow(ink, cmap=\'gray_r\', vmin=0, vmax=1, alpha=.2)\n            axes[1, col].imshow(np.ma.masked_where(~diagnostic[\'skeleton\'], diagnostic[\'skeleton\']),\n                                cmap=\'gray_r\', vmin=0, vmax=1)\n            if diagnostic[\'centers\']:\n                yy, xx = np.asarray(diagnostic[\'centers\']).T\n                axes[1, col].scatter(xx, yy, s=100, facecolors=\'none\', edgecolors=\'red\', linewidths=1.8)\n            if diagnostic[\'uncertain_centers\']:\n                yy, xx = np.asarray(diagnostic[\'uncertain_centers\']).T\n                axes[1, col].scatter(xx, yy, s=75, facecolors=\'none\', edgecolors=\'#d99000\', linewidths=1.4)\n            axes[1, col].set_title(f"Graph check: {diagnostic[\'branch_regions_max\']} resolved; "\n                f"{diagnostic[\'unresolved_regions_max\']} unresolved max\\n"\n                f"shown threshold={diagnostic[\'selected_threshold\']:.2f}; "\n                f"hits={diagnostic[\'levels_with_branches\']}/{diagnostic[\'tested_levels\']}", fontsize=10)\n            response = item[\'junction\'][0, 0].cpu().numpy()\n            axes[2, col].imshow(ink, cmap=\'gray_r\', vmin=0, vmax=1, alpha=.2)\n            artist = axes[2, col].imshow(np.ma.masked_less_equal(response, 0), cmap=\'magma\', vmin=0, vmax=display_max)\n            fig.colorbar(artist, ax=axes[2, col], fraction=.045, pad=.02)\n            axes[2, col].set_title(f"Soft junction response: J={values[\'junction\']:.3g}\\n"\n                                  f"peak={values[\'junction_peak_response\']:.3g}", fontsize=10)\n            for ax in axes[:, col]:\n                ax.axis(\'off\')\n        # ASCII fallback avoids font warnings hiding the actual diagnostic output.\n        safe_caption = unicodedata.normalize(\'NFC\', caption).encode(\'ascii\', \'backslashreplace\').decode(\'ascii\')\n        fig.suptitle(safe_caption + \'\\nRed = resolved persistent branches; amber = unresolved. Soft heat uses a shared PAGE scale.\', fontsize=10)\n        path = str(prefix) + f\'_junction_{offset//4+1}.png\'\n        fig.savefig(path, dpi=140)\n        paths.append(path)\n        if show:\n            plt.show()\n        plt.close(fig)\n    return paths\n\n\ndef _plot(maps, rows, prefix, caption, show):\n    import matplotlib.pyplot as plt\n    import unicodedata\n    paths = []\n    for offset in range(0, len(maps), 4):\n        page = maps[offset:offset + 4]\n        fig, axes = plt.subplots(5, len(page), squeeze=False, figsize=(3.2*len(page), 12))\n        for col, item in enumerate(page):\n            k = offset + col\n            for row, (name, title) in enumerate([\n                    (\'centerline\', \'Reference centerline\'), (\'mask\', \'Soft mask (absolute ink)\'),\n                    (\'missing\', \'Missing on reference path\'),\n                    (\'split\', \'Extractor connectivity diagnostic\'), (\'junction\', \'Extractor junction defect\')]):\n                content = item[name][0, 0].cpu().numpy()\n                ax = axes[row, col]\n                if row >= 2:\n                    ax.imshow(item[\'mask\'][0, 0].cpu().numpy(), cmap=\'gray_r\', vmin=0, vmax=1, alpha=.25)\n                    artist = ax.imshow(np.ma.masked_less_equal(content, 1e-8), cmap=\'magma\',\n                                       vmin=0, vmax=1 if name != \'junction\' else max(1., float(content.max())))\n                    fig.colorbar(artist, ax=ax, fraction=.04, pad=.02)\n                else:\n                    ax.imshow(content, cmap=\'gray_r\', vmin=0, vmax=1)\n                ax.set_title(f\'Stroke {k+1}: {title}\', fontsize=9)\n                ax.axis(\'off\')\n            values = rows[k]\n            axes[4, col].text(.5, -.08,\n                f"M={values[\'missing\']:.4f}  D={values[\'split\']:.4f}\\n"\n                f"J={values[\'junction\']:.4f}  W={values[\'width\']:.4f}",\n                transform=axes[4, col].transAxes, ha=\'center\', va=\'top\', fontsize=9)\n        safe_caption = unicodedata.normalize(\'NFC\', caption).encode(\'ascii\', \'backslashreplace\').decode(\'ascii\')\n        fig.suptitle(safe_caption + \'\\nExtractor defects + reference deficits. Source-image quality is NOT assessed.\', fontsize=10)\n        fig.tight_layout(rect=(0, .025, 1, .96))\n        path = str(prefix) + f\'_structure_{offset//4+1}.png\'\n        fig.savefig(path, dpi=140)\n        paths.append(path)\n        if show:\n            plt.show()\n        plt.close(fig)\n    return paths\n\n\ndef score_saved_strokes(npz_paths, config=None, save_dir=None, show=True, show_details=False):\n    """Use saved run_inference_csv NPZ files: soft_masks + reference_strokes.\n\n    Also accepts the inference report dictionary returned by run_inference_csv.\n    Preserves reference_geometry if saved; otherwise uses reference_strokes.\n    Input and geometry must already be aligned.\n    """\n    config = config or StructureConfig()\n    if isinstance(npz_paths, dict):\n        npz_paths = [item[\'npz\'] for item in npz_paths[\'results\']]\n    elif isinstance(npz_paths, (str, Path)):\n        npz_paths = [npz_paths]\n    npz_paths = list(npz_paths)\n    if not npz_paths:\n        raise ValueError(\'No NPZ files selected\')\n    destination = _destination(save_dir)\n    results = []\n    for i, path in enumerate(npz_paths):\n        with np.load(path, allow_pickle=False) as data:\n            masks = torch.tensor(data[\'soft_masks\'], dtype=torch.float32)\n            reference = torch.tensor(data[\'reference_strokes\'], dtype=torch.float32)\n            geometry = torch.tensor(data[\'reference_geometry\'], dtype=torch.float32) if \'reference_geometry\' in data else reference\n            target_image = (data[\'reference_image\'].copy() if \'reference_image\' in data else\n                            data[\'reference\'].copy() if \'reference\' in data else None)\n            image = data[\'image\'].copy() if \'image\' in data else None\n        if masks.ndim == 4 and masks.shape[0] == 1:\n            masks = masks[0]\n        if masks.ndim != 3:\n            raise ValueError(f\'Expected K,H,W masks in {path}\')\n        results.append({\'input_npz\': str(path), **_measure(masks, geometry, config,\n            destination / f\'sample_{i+1:04d}\', reference=reference, image=image,\n            show=show, caption=Path(path).name, show_details=show_details, target_image=target_image)})\n    report = {\'mode\': \'saved_npz\', \'geometry\': \'stored_if_available_else_reference\', \'config\': asdict(config),\n              \'source_quality\': {\'assessed\': False}, \'results\': results}\n    (destination / \'structure_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Structure results:\', destination)\n    return report\n\n\ndef score_structure_csv(checkpoint, csv_path, input_files=None, sample_rows=None,\n                        dataset_root=None, config=None, geometry=\'reference\',\n                        save_dir=None, show=True, show_details=False):\n    """Predict selected PNGs then measure errors, ignoring target annotations.\n\n    geometry=\'reference\': fixed reference coordinates, requires alignment.\n    geometry=\'hints\': detached model-warped reference; can hide alignment errors.\n    Evaluation only. For guidance call StrokeStructureLoss on non-detached masks.\n    """\n    from stroke_model.convert_reference_csv import Resolver, existing_path\n    from stroke_model.infer_reference import inference_columns, load_inference_row, select_input_files\n    from stroke_model.reference_model import ReferenceStrokeModel\n\n    if geometry not in (\'reference\', \'hints\'):\n        raise ValueError(\'geometry must be reference or hints\')\n    if input_files is not None and sample_rows is not None:\n        raise ValueError(\'Specify input_files or sample_rows, not both\')\n    config = config or StructureConfig()\n    checkpoint, csv_path = existing_path(checkpoint), existing_path(csv_path)\n    frame = pd.read_csv(csv_path)\n    columns = inference_columns(frame)\n    selected = (select_input_files(frame, columns[\'image\'], input_files) if input_files is not None else\n                list(range(1, len(frame)+1)) if sample_rows is None else list(sample_rows))\n    if not selected or any(not isinstance(i, int) or not 1 <= i <= len(frame) for i in selected):\n        raise ValueError(f\'Select CSV rows in 1..{len(frame)}\')\n    resolve = Resolver(csv_path, existing_path(dataset_root) if dataset_root else csv_path.parent)\n    state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n    model = ReferenceStrokeModel(image_size=state[\'image_size\'], width=state[\'width\'], pretrained=False)\n    model.load_state_dict(state[\'model\'], strict=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = model.to(device).eval()\n    size = state[\'image_size\']\n    del state\n    destination = _destination(save_dir or checkpoint.parent / (\'structure_\' + datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\')))\n    print(\'Checkpoint:\', checkpoint)\n    print(\'CSV:\', csv_path)\n    print(\'Reference geometry:\', geometry, \'/ annotations are not used\')\n    results = []\n    for number in selected:\n        image, reference, strokes, image_path, _ = load_inference_row(frame.iloc[number-1], columns, resolve, size)\n        with torch.no_grad():\n            output = model(image[None].to(device), reference[None].to(device), strokes[None].to(device))\n        # Reference preparation uses CPU only; scalar scoring runs on model device.\n        prepared_geometry = strokes if geometry == \'reference\' else output[\'hints\'][0].detach().cpu()\n        result = _measure(output[\'masks\'][0], prepared_geometry, config,\n            destination / f\'row_{number:04d}\', reference=strokes, image=image,\n            show=show, caption=f\'CSV row {number}: {image_path.name}\', show_details=show_details,\n            target_image=reference)\n        results.append({\'csv_row\': number, \'input\': str(image_path), **result})\n    report = {\'mode\': \'csv\', \'checkpoint\': str(checkpoint), \'csv\': str(csv_path),\n              \'geometry\': geometry, \'config\': asdict(config), \'uses_input_stroke_annotations\': False,\n              \'source_quality\': {\'assessed\': False}, \'results\': results}\n    (destination / \'structure_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Structure results:\', destination)\n    return report\n'}
    import types
    for _name, _source in _indicator_sources.items():
        _module = types.ModuleType(_name)
        _module.__package__ = "stroke_model"
        _module.__file__ = "<embedded:" + _name + ">"
        sys.modules[_name] = _module
        exec(compile(_source, _module.__file__, "exec"), _module.__dict__)


In [ ]:
# 점수 모듈 내장: 추가 모델·가중치 다운로드 없이 실행
import sys, types
_extra_module = types.ModuleType("_extra_ink_runtime")
_extra_module.__file__ = "<embedded:extra_ink>"
sys.modules[_extra_module.__name__] = _extra_module
exec(compile('"""Reference-aware unexplained ink, measured on the INPUT, not assigned masks.\n\nReference centerlines define tolerant support regions with bounded translations. This\ndetects both attached and detached added ink even if every input pixel was\nassigned to an existing stroke by softmax. It is not an extra-stroke counter.\n"""\nimport math\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt, shift\nfrom skimage.morphology import skeletonize\nimport torch\nfrom torch import nn\n\n\nclass ExtraInkMetric(nn.Module):\n    def __init__(self, aligned_reference_strokes, radius=8., transition=0.5,\n                 min_ink_mass=1., reference_threshold=0.5, alignment=6,\n                 alignment_step=2, alignment_temperature=.01, alignment_prior=.001,\n                 alignment_sigma=2.):\n        super().__init__()\n        if not all(math.isfinite(x) for x in (radius, transition, min_ink_mass, reference_threshold)):\n            raise ValueError(\'Extra ink settings must be finite\')\n        if radius < 0 or transition <= 0 or min_ink_mass <= 0 or not 0 < reference_threshold < 1:\n            raise ValueError(\'Invalid extra ink radius, transition, mass or threshold\')\n        if (not isinstance(alignment, int) or alignment < 0 or not isinstance(alignment_step, int)\n                or alignment_step < 1 or not math.isfinite(alignment_temperature)\n                or alignment_temperature <= 0 or not math.isfinite(alignment_prior) or alignment_prior < 0\n                or not math.isfinite(alignment_sigma) or alignment_sigma <= 0):\n            raise ValueError(\'Invalid bounded alignment settings\')\n        reference = torch.as_tensor(aligned_reference_strokes).detach().float().cpu().numpy()\n        if reference.ndim != 3 or not np.isfinite(reference).all() or (reference < 0).any() or (reference > 1).any():\n            raise ValueError(\'Reference must be finite K,H,W ink in [0,1]\')\n        if not len(reference) or any(not (s > reference_threshold).any() for s in reference):\n            raise ValueError(\'Keep every expected reference stroke; empty reference strokes are invalid\')\n        # Fixed translation banks; prediction-dependent weights stay in torch.\n        offsets = sorted(set(range(-alignment, alignment+1, alignment_step)) | {0, alignment})\n        coords = [(dy, dx) for dy in offsets for dx in offsets]\n        matches, supports = [], []\n        for stroke in reference:\n            distance = distance_transform_edt(~skeletonize(stroke > reference_threshold))\n            match = np.exp(-.5*(distance/alignment_sigma)**2)\n            support = 1/(1+np.exp(np.clip((distance-radius)/transition, -80, 80)))\n            match_bank, support_bank = [], []\n            for dy, dx in coords:\n                moved = shift(match, (dy, dx), order=0, mode=\'constant\')\n                match_bank.append(moved/max(moved.sum(), 1e-8))\n                support_bank.append(shift(support, (dy, dx), order=0, mode=\'constant\'))\n            matches.append(match_bank)\n            supports.append(support_bank)\n        self.register_buffer(\'matches\', torch.tensor(np.asarray(matches), dtype=torch.float32))\n        self.register_buffer(\'supports\', torch.tensor(np.asarray(supports), dtype=torch.float32))\n        self.register_buffer(\'shift_cost\', torch.tensor([alignment_prior*(dy*dy+dx*dx)\n                                                       for dy, dx in coords], dtype=torch.float32))\n        self.alignment_temperature = alignment_temperature\n        self.min_ink_mass = min_ink_mass\n\n    def forward(self, image):\n        image = torch.as_tensor(image)\n        if image.ndim == 2:\n            image = image[None, None]\n        elif image.ndim == 3:\n            image = image[None]\n        if image.ndim != 4 or image.shape[1] != 1 or image.shape[-2:] != self.supports.shape[-2:]:\n            raise ValueError(\'image must have shape H,W or 1,H,W or B,1,H,W matching the reference\')\n        if not image.is_floating_point():\n            image = image.float()\n        if image.device != self.supports.device:\n            raise ValueError(\'Move ExtraInkMetric to the image device\')\n        if not bool(torch.isfinite(image).all()) or bool((image < 0).any()) or bool((image > 1).any()):\n            raise ValueError(\'Image must be finite absolute ink in [0,1]\')\n        matches = self.matches.to(dtype=image.dtype)\n        supports = self.supports.to(dtype=image.dtype)\n        alignment_scores = torch.einsum(\'bchw,knhw->bkn\', image, matches)-self.shift_cost\n        weights = torch.softmax(alignment_scores/self.alignment_temperature, dim=-1)\n        per_stroke_support = torch.einsum(\'bkn,knhw->bkhw\', weights, supports)\n        support = 1-(1-per_stroke_support).prod(1, keepdim=True)\n        extra_map = image * (1-support)\n        extra_mass = extra_map.flatten(1).sum(1)\n        fraction = extra_mass / image.flatten(1).sum(1).clamp_min(self.min_ink_mass)\n        return {\'extra\': fraction, \'extra_map\': extra_map, \'support\': support, \'alignment_weights\': weights,\n                \'extra_mass\': extra_mass}\n', _extra_module.__file__, "exec"), _extra_module.__dict__)
_score_module = types.ModuleType("_structural_indicator_runtime")
_score_module.__file__ = "<embedded:structural_indicator>"
sys.modules[_score_module.__name__] = _score_module
exec(compile('"""Deterministic structural indicator with explicit added-ink penalty.\n\nNo survey, fitted parameters, image embeddings, width or hard branch counts.\nThe 0..100 scale is a configurable engineering index, not calibrated quality.\n"""\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime\nimport json\nimport math\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom _extra_ink_runtime import ExtraInkMetric\n\n\n@dataclass(frozen=True)\nclass IndicatorConfig:\n    missing_weight: float = 1.0\n    split_weight: float = 1.0\n    junction_weight: float = 1.0\n    split_scale: float = 0.2\n    junction_scale: float = 0.01\n    worst_stroke_weight: float = 0.5\n    priority_mode: bool = True\n    split_max_penalty: float = 0.60\n    junction_max_penalty: float = 0.65\n    extra_weight: float = 1.0\n    extra_scale: float = 0.025\n    extra_allowance: float = 0.03\n    extra_radius: float = 8.0\n    extra_transition: float = 0.5\n    extra_alignment: int = 6\n    extra_alignment_step: int = 2\n    extra_alignment_temperature: float = 0.01\n    extra_alignment_prior: float = 0.001\n    extra_alignment_sigma: float = 2.0\n    eps: float = 1e-6\n\n    def __post_init__(self):\n        for name, value in asdict(self).items():\n            if not math.isfinite(value):\n                raise ValueError(f\'{name} must be finite\')\n        if min(self.missing_weight, self.split_weight, self.junction_weight, self.extra_weight) < 0:\n            raise ValueError(\'Weights must be nonnegative\')\n        if self.split_scale <= 0 or self.junction_scale <= 0 or not 0 < self.eps < 1:\n            raise ValueError(\'Scales must be positive; eps must be in (0,1)\')\n        if self.extra_scale <= 0 or not 0 <= self.extra_allowance < 1 or self.extra_radius < 0 or self.extra_transition <= 0:\n            raise ValueError(\'Invalid extra-ink scale, allowance or geometry settings\')\n        for weight, scale in ((self.split_weight, self.split_scale), (self.junction_weight, self.junction_scale), (self.extra_weight, self.extra_scale)):\n            if not math.isfinite(weight/scale) or weight/scale > torch.finfo(torch.float32).max:\n                raise ValueError(\'weight/scale must fit a finite float32 coefficient\')\n        if not 0 <= self.worst_stroke_weight <= 1:\n            raise ValueError(\'worst_stroke_weight must be in [0,1]\')\n        if not isinstance(self.priority_mode, bool):\n            raise ValueError(\'priority_mode must be bool\')\n        if not 0 <= self.split_max_penalty <= self.junction_max_penalty < 1:\n            raise ValueError(\'Require 0 <= split_max_penalty <= junction_max_penalty < 1\')\n        if (not isinstance(self.extra_alignment, int) or self.extra_alignment < 0\n                or not isinstance(self.extra_alignment_step, int) or self.extra_alignment_step < 1\n                or self.extra_alignment_temperature <= 0 or self.extra_alignment_prior < 0\n                or self.extra_alignment_sigma <= 0):\n            raise ValueError(\'Invalid bounded alignment settings\')\n\n\nclass StructuralIndicator(nn.Module):\n    """Reference error scores with configurable severity priorities.\n\n    Priority mode bounds isolated split/junction penalties and caps the glyph\n    score by its worst missing stroke and unexplained ink. Combined defects\n    can still score below the isolated-error bands. min uses subgradients.\n    priority_mode=False reproduces the legacy exponential aggregation.\n    """\n    def __init__(self, config=None):\n        super().__init__()\n        self.config = config or IndicatorConfig()\n\n    def forward(self, missing, split, junction, active=None, *, extra=None):\n        missing = torch.as_tensor(missing)\n        if not missing.is_floating_point():\n            missing = missing.float()\n        split = torch.as_tensor(split, device=missing.device, dtype=missing.dtype)\n        junction = torch.as_tensor(junction, device=missing.device, dtype=missing.dtype)\n        if missing.ndim not in (1, 2) or missing.shape != split.shape or missing.shape != junction.shape:\n            raise ValueError(\'missing, split, junction must have matching K or B,K shapes\')\n        if active is None:\n            active = torch.ones_like(missing, dtype=torch.bool)\n        if active.dtype != torch.bool or active.shape != missing.shape or active.device != missing.device:\n            raise ValueError(\'active must be a bool tensor matching the input shape/device\')\n        if missing.ndim == 1:\n            missing, split, junction, active = [x[None] for x in (missing, split, junction, active)]\n        for name, value in [(\'missing\', missing), (\'split\', split), (\'junction\', junction)]:\n            if not bool(torch.isfinite(value).all()) or bool((value < 0).any()):\n                raise ValueError(f\'{name} must be finite and nonnegative\')\n        if bool((missing > 1).any()) or not bool(active.any(1).all()):\n            raise ValueError(\'Missing must be <=1 and each sample needs an active stroke\')\n        cfg = self.config\n        # Smooth endpoint extension: exact 0/1 retention with finite gradients,\n        # including fractional weights. Unlike a hard zero branch, full missing\n        # still receives a restorative gradient at the default weight of one.\n        if cfg.missing_weight == 0:\n            retain_m = 1 + missing*0\n        else:\n            epsilon = missing.new_tensor(cfg.eps)\n            lower = epsilon.pow(cfg.missing_weight)\n            denominator = (1+epsilon).pow(cfg.missing_weight)-lower\n            retain_m = ((1-missing+epsilon).pow(cfg.missing_weight)-lower)/denominator\n        retain_d = torch.exp(-(cfg.split_weight/cfg.split_scale) * split)\n        retain_j = torch.exp(-(cfg.junction_weight/cfg.junction_scale) * junction)\n        if cfg.priority_mode:\n            d = (cfg.split_weight/cfg.split_scale)*split\n            j = (cfg.junction_weight/cfg.junction_scale)*junction\n            retain_d = 1-cfg.split_max_penalty*(1-1/(1+d))\n            retain_j = 1-cfg.junction_max_penalty*(1-1/(1+j))\n        stroke_scores = 100 * retain_m * retain_d * retain_j\n        mean = (stroke_scores * active).sum(1) / active.sum(1)\n        worst = stroke_scores.masked_fill(~active, torch.inf).amin(1)\n        base_score = (1-cfg.worst_stroke_weight)*mean + cfg.worst_stroke_weight*worst\n        cap = lambda retain: 100*retain.masked_fill(~active, torch.inf).amin(1)\n        missing_cap, split_cap, junction_cap = [cap(r) for r in (retain_m, retain_d, retain_j)]\n        if cfg.priority_mode:\n            base_score = torch.stack([base_score, missing_cap, split_cap, junction_cap]).amin(0)\n        if extra is None:\n            if cfg.extra_weight > 0:\n                raise ValueError(\'Added ink is required: provide extra or explicitly set extra_weight=0\')\n            extra = base_score*0\n        extra = torch.as_tensor(extra, device=missing.device, dtype=missing.dtype)\n        if extra.ndim == 0 and len(base_score) == 1:\n            extra = extra[None]\n        if extra.shape != base_score.shape or not bool(torch.isfinite(extra).all()) or bool(((extra < 0) | (extra > 1)).any()):\n            raise ValueError(\'extra must contain one finite [0,1] fraction per glyph\')\n        extra_penalty = torch.relu(extra-cfg.extra_allowance)\n        extra_retention = torch.exp(-(cfg.extra_weight/cfg.extra_scale)*extra_penalty)\n        extra_cap = 100*extra_retention\n        score = torch.minimum(base_score, extra_cap) if cfg.priority_mode else base_score * extra_retention\n        return {\'score\': score, \'stroke_scores\': stroke_scores, \'mean_score\': mean,\n                \'worst_score\': worst, \'penalty\': 1-score/100, \'active\': active,\n                \'base_score\': base_score, \'extra\': extra, \'extra_retention\': extra_retention,\n                \'caps\': {\'missing\': missing_cap, \'split\': split_cap, \'junction\': junction_cap, \'extra\': extra_cap},\n                \'retention\': {\'missing\': retain_m, \'split\': retain_d, \'junction\': retain_j}}\n\n\ndef _extra_for_sample(sample, source_path, config):\n    """Old reports are remeasured from snapshots; never assume missing extra=0."""\n    import numpy as np\n    if \'extra\' in sample:\n        return float(sample[\'extra\']), None, \'provided_value\'\n    if config.extra_weight == 0:\n        return 0., None, \'disabled\'\n    for key in (\'npz\', \'input_npz\'):\n        value = sample.get(key)\n        if not value:\n            continue\n        paths = [Path(value)]\n        if source_path and not paths[0].is_absolute():\n            paths.append(source_path.parent/paths[0])\n        for path in paths:\n            if not path.is_file():\n                continue\n            with np.load(path, allow_pickle=False) as arrays:\n                target_key = next((key for key in (\'reference_image\', \'reference\', \'reference_geometry\', \'reference_strokes\')\n                                   if key in arrays), None)\n                if \'image\' not in arrays or target_key not in arrays:\n                    continue\n                image = torch.tensor(arrays[\'image\'], dtype=torch.float32)\n                target = torch.tensor(arrays[target_key], dtype=torch.float32)\n            metric = ExtraInkMetric(target, radius=config.extra_radius, transition=config.extra_transition,\n                alignment=config.extra_alignment, alignment_step=config.extra_alignment_step,\n                alignment_temperature=config.extra_alignment_temperature, alignment_prior=config.extra_alignment_prior,\n                alignment_sigma=config.extra_alignment_sigma)\n            with torch.no_grad():\n                result = metric(image)\n            return float(result[\'extra\'][0]), {\'image\': image.squeeze().numpy(),\n                \'extra_map\': result[\'extra_map\'][0, 0].numpy(),\n                \'support\': result[\'support\'][0, 0].numpy(),\n                \'alignment_weights\': result[\'alignment_weights\'][0].numpy()}, f\'{path}::{target_key}\'\n    raise ValueError(\'Cannot assess added ink: keep the NPZ containing image and reference_geometry/reference_strokes. \'\n                     \'Do not treat absent extra measurements as zero; extra_weight=0 explicitly opts out.\')\n\n\ndef _image_for_sample(sample, source_path):\n    """Prefer the scored input snapshot; never substitute a reference/montage."""\n    import numpy as np\n    from PIL import Image, ImageOps\n\n    def candidates(value):\n        if not value:\n            return []\n        path = Path(value)\n        choices = [path]\n        if not path.is_absolute() and source_path:\n            choices.append(source_path.parent/path)\n        return choices\n\n    for key in (\'npz\', \'input_npz\'):\n        for path in candidates(sample.get(key)):\n            if not path.is_file() or path.suffix.lower() != \'.npz\':\n                continue\n            with np.load(path, allow_pickle=False) as data:\n                if \'image\' not in data:\n                    continue\n                ink = np.asarray(data[\'image\'], dtype=np.float32)\n            if ink.ndim == 3 and ink.shape[0] == 1:\n                ink = ink[0]\n            if ink.ndim != 2 or not np.isfinite(ink).all() or (ink < 0).any() or (ink > 1).any():\n                continue\n            # Saved arrays use ink=1. Display black ink on a white background.\n            content = np.rint(255*(1-ink)).astype(np.uint8)\n            return Image.fromarray(content).convert(\'RGB\'), str(path), \'scored_input_array\'\n    for path in candidates(sample.get(\'input\')):\n        if path.suffix.lower() in (\'.png\', \'.jpg\', \'.jpeg\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\') and path.is_file():\n            with Image.open(path) as image:\n                return ImageOps.exif_transpose(image).convert(\'RGB\'), str(path), \'original_file\'\n    return None, None, \'unavailable\'\n\n\ndef _score_gallery(samples, scored, destination, source_path, columns, per_page, show):\n    import base64\n    import html\n    import io\n    import unicodedata\n    import matplotlib.pyplot as plt\n\n    cards, loaded, pngs = [], [], []\n    for number, (sample, result) in enumerate(zip(samples, scored), 1):\n        image, path, kind = _image_for_sample(sample, source_path)\n        result.update(image_available=image is not None, image_source=path, image_kind=kind)\n        name = str(sample.get(\'input\') or sample.get(\'input_npz\') or f\'Image {number}\').replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1]\n        loaded.append((image, name, result))\n        if image is None:\n            picture = \'<div class="indicator-no-image">사진 경로를 찾지 못했습니다</div>\'\n        else:\n            buffer = io.BytesIO()\n            image.save(buffer, format=\'PNG\')\n            encoded = base64.b64encode(buffer.getvalue()).decode(\'ascii\')\n            picture = f\'<img src="data:image/png;base64,{encoded}" alt="{html.escape(name, quote=True)}">\'\n        detail_rows = \'\'.join(f"<tr><td>{html.escape(str(s[\'stroke\']))}</td><td>{s[\'stroke_score\']:.1f}</td>"\n                              f"<td>{s[\'missing\']:.3g}</td><td>{s[\'split\']:.3g}</td><td>{s[\'junction\']:.3g}</td></tr>"\n                              for s in result[\'strokes\'])\n        note = \'평가에 사용한 입력 배열\' if kind == \'scored_input_array\' else \'입력 사진\'\n        extra_detail = \'\'\n        if result.get(\'extra_overlay\'):\n            encoded = base64.b64encode(Path(result[\'extra_overlay\']).read_bytes()).decode(\'ascii\')\n            extra_detail = (\'<details><summary>추가 잉크 위치 보기</summary>\'\n                f\'<img src="data:image/png;base64,{encoded}" alt="추가 잉크 위치">\'\n                \'<p>빨간색은 기준 허용 영역 밖의 잉크입니다.</p></details>\')\n        extra_note = (f"추가 잉크 {100*result[\'extra\']:.1f}% · 추가 감점 전 {result[\'score_before_extra\']:.1f}점"\n                      if result[\'extra_assessed\'] else \'추가 잉크 검사 꺼짐\')\n        cap_rows = \'\'.join(f"<tr><td>{label}</td><td>{result[key+\'_score_cap\']:.1f}</td></tr>"\n                           for key, label in [(\'missing\', \'누락\'), (\'extra\', \'추가\'),\n                                              (\'junction\', \'정션\'), (\'split\', \'분리\')])\n        cap_detail = (\'<details><summary>오류별 점수 상한 보기</summary><table>\'\n                      + cap_rows + \'</table><p>복합 오류는 상한보다 더 낮아질 수 있습니다.</p></details>\')\n        if not result[\'priority_mode\']:\n            cap_detail = \'<p>이전 지수 곱셈 공식: 점수 상한 미적용</p>\'\n        cards.append(f\'<article class="indicator-card">{picture}\'\n                     f\'<div class="indicator-name">{number}. {html.escape(name)}</div>\'\n                     f\'<div class="indicator-score">{result["structural_score"]:.1f}<span> / 100</span></div>\'\n                     f\'<div class="indicator-note">{note if image is not None else "이미지 없는 수치 입력"}</div>\'\n                     f\'<div class="indicator-note">{extra_note}</div>{extra_detail}{cap_detail}\'\n                     \'<details><summary>획별 점수 보기</summary><table><thead><tr><th>획</th><th>점수</th>\'\n                     f\'<th>누락</th><th>분리</th><th>정션</th></tr></thead><tbody>{detail_rows}</tbody></table></details></article>\')\n    style = f\'\'\'<style>\n.indicator-gallery {{font-family:system-ui,sans-serif;color:#202124;}}\n.indicator-grid {{display:grid;grid-template-columns:repeat({columns},minmax(0,1fr));gap:16px;}}\n.indicator-card {{border:1px solid #ddd;border-radius:12px;padding:16px;background:white;min-width:0;}}\n.indicator-card img,.indicator-no-image {{width:100%;height:240px;object-fit:contain;background:white;}}\n.indicator-no-image {{display:flex;align-items:center;justify-content:center;color:#777;}}\n.indicator-name {{font-size:14px;overflow-wrap:anywhere;margin:10px 0 4px;}}\n.indicator-score {{font-size:36px;font-weight:700;}} .indicator-score span {{font-size:16px;font-weight:400;color:#666;}}\n.indicator-note {{font-size:12px;color:#666;margin:4px 0 12px;}}\n.indicator-gallery table {{font-size:12px;border-collapse:collapse;width:100%;margin-top:10px;}}\n.indicator-gallery td,.indicator-gallery th {{padding:5px;text-align:right;border-bottom:1px solid #eee;}}\n@media(max-width:650px){{.indicator-grid{{grid-template-columns:repeat(2,minmax(0,1fr));}}}}\n</style>\'\'\'\n    fragment = style + \'<section class="indicator-gallery"><p>사진별 구조 점수 · 누락·분리·정션·추가 잉크를 감점합니다.</p>\'\n    fragment += \'<div class="indicator-grid">\' + \'\'.join(cards) + \'</div></section>\'\n    gallery_path = destination/\'images_and_scores.html\'\n    gallery_path.write_text(\'<!doctype html><html lang="ko"><meta charset="utf-8">\'+fragment+\'</html>\', encoding=\'utf-8\')\n    for offset in range(0, len(loaded), per_page):\n        page = loaded[offset:offset+per_page]\n        cols = min(columns, len(page))\n        rows = math.ceil(len(page)/cols)\n        figure, axes = plt.subplots(rows, cols, squeeze=False, figsize=(3.5*cols, 4.2*rows), layout=\'constrained\')\n        for ax in axes.flat:\n            ax.axis(\'off\')\n        for ax, (image, name, result) in zip(axes.flat, page):\n            if image is not None:\n                ax.imshow(image)\n            else:\n                ax.text(.5, .5, \'Image unavailable\', transform=ax.transAxes, ha=\'center\', color=\'#777\')\n            safe_name = unicodedata.normalize(\'NFC\', name).encode(\'ascii\', \'backslashreplace\').decode(\'ascii\')\n            if len(safe_name) > 58:\n                safe_name = safe_name[:55]+\'...\'\n            ax.set_title(f\'{result["sample"]}. {safe_name}\\n{result["structural_score"]:.1f} / 100\', fontsize=12)\n        figure.suptitle(\'Image : structural score (0..100)\', fontsize=14)\n        path = destination/f\'images_and_scores_{offset//per_page+1:02d}.png\'\n        figure.savefig(path, dpi=150)\n        plt.close(figure)\n        pngs.append(str(path))\n    if show:\n        try:\n            from IPython import get_ipython\n            from IPython.display import HTML, display\n            if get_ipython() is not None:\n                display(HTML(fragment))\n        except ImportError:\n            pass\n    return str(gallery_path), pngs\n\n\ndef score_structure_report(report, config=None, save_dir=None, *, show_images=True,\n                           images_per_row=3, images_per_page=12, show_stroke_table=False):\n    """Score an existing structure_report.json/dict without rerunning extraction.\n\n    Returns an auditable report and saves glyph_scores.csv, stroke_scores.csv,\n    indicator_report.json. Uses M/D/J and remeasured extra ink; no filename-based scoring.\n    """\n    import pandas as pd\n\n    if any(not isinstance(v, int) or v < 1 for v in (images_per_row, images_per_page)):\n        raise ValueError(\'Gallery row/page counts must be positive integers\')\n    cfg = config or IndicatorConfig()\n    source_path = None\n    if isinstance(report, (str, Path)):\n        source_path = Path(report)\n        report = json.loads(source_path.read_text(encoding=\'utf-8-sig\'))\n    if not isinstance(report, dict) or not report.get(\'results\'):\n        raise ValueError(\'Provide structure_results or a structure_report.json with nonempty results\')\n    if save_dir is None:\n        base = source_path.parent if source_path else Path(report[\'results\'][0].get(\'npz\', \'.\')).parent\n        save_dir = base / (\'indicator_\' + datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    destination = Path(save_dir)\n    destination.mkdir(parents=True, exist_ok=False)\n    scorer = StructuralIndicator(cfg)\n    glyph_rows, stroke_rows, results = [], [], []\n    for number, sample in enumerate(report[\'results\'], 1):\n        strokes = sample.get(\'strokes\', [])\n        if not strokes:\n            raise ValueError(f\'Sample {number} has no per-stroke measurements\')\n        missing, split, junction = [torch.tensor([s[key] for s in strokes], dtype=torch.float64)\n                                     for key in (\'missing\', \'split\', \'junction\')]\n        extra, extra_maps, extra_source = _extra_for_sample(sample, source_path, cfg)\n        with torch.no_grad():\n            output = scorer(missing, split, junction, extra=[extra])\n        identity = {\'sample\': number, \'csv_row\': sample.get(\'csv_row\'),\n                    \'input\': sample.get(\'input\') or sample.get(\'input_npz\') or \'\'}\n        glyph = {**identity, \'structural_score\': float(output[\'score\'][0]),\n                 \'priority_mode\': cfg.priority_mode,\n                 \'score_before_extra\': float(output[\'base_score\'][0]), \'extra\': extra,\n                 \'extra_retention\': float(output[\'extra_retention\'][0]),\n                 \'extra_assessed\': extra_source != \'disabled\',\n                 **{name+\'_score_cap\': float(value[0]) for name, value in output[\'caps\'].items()},\n                 \'mean_stroke_score\': float(output[\'mean_score\'][0]),\n                 \'worst_stroke_score\': float(output[\'worst_score\'][0]),\n                 \'stroke_count\': len(strokes)}\n        glyph_rows.append(glyph)\n        scored_strokes = []\n        for k, stroke in enumerate(strokes):\n            row = {**identity, \'stroke\': stroke.get(\'stroke\', k+1),\n                   \'missing\': float(missing[k]), \'split\': float(split[k]), \'junction\': float(junction[k]),\n                   \'stroke_score\': float(output[\'stroke_scores\'][0, k]),\n                   **{name+\'_retention\': float(value[0, k]) for name, value in output[\'retention\'].items()}}\n            stroke_rows.append(row)\n            scored_strokes.append(row)\n        extra_overlay = None\n        if extra_maps is not None:\n            import numpy as np\n            from PIL import Image\n            np.savez_compressed(destination/f\'extra_{number:04d}.npz\', **extra_maps)\n            background = np.repeat((1-extra_maps[\'image\'])[..., None], 3, axis=-1)\n            alpha = np.clip(extra_maps[\'extra_map\'], 0, 1)[..., None]\n            colored = background*(1-alpha) + np.array([1., 0., 0.])*alpha\n            extra_overlay = destination/f\'extra_{number:04d}.png\'\n            Image.fromarray(np.rint(colored*255).astype(np.uint8)).save(extra_overlay)\n        results.append({**glyph, \'extra_source\': extra_source,\n                        \'extra_overlay\': str(extra_overlay) if extra_overlay else None, \'strokes\': scored_strokes})\n    pd.DataFrame(glyph_rows).to_csv(destination / \'glyph_scores.csv\', index=False, encoding=\'utf-8-sig\')\n    pd.DataFrame(stroke_rows).to_csv(destination / \'stroke_scores.csv\', index=False, encoding=\'utf-8-sig\')\n    gallery, gallery_images = _score_gallery(report[\'results\'], results, destination, source_path,\n                                             images_per_row, images_per_page, show_images)\n    output_report = {\'indicator\': \'structural_indicator_v3_priority\', \'range\': [0, 100],\n                     \'uses_only\': [\'missing\', \'split\', \'junction\'] + ([\'extra\'] if cfg.extra_weight > 0 else []), \'trained\': False,\n                     \'calibrated_to_human_quality\': False, \'config\': asdict(cfg),\n                     \'measurement_config\': report.get(\'config\'), \'source_report\': str(source_path) if source_path else None,\n                     \'save_dir\': str(destination), \'gallery_html\': gallery,\n                     \'gallery_images\': gallery_images, \'results\': results}\n    (destination / \'indicator_report.json\').write_text(json.dumps(output_report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Structural indicator (0..100; higher is better; uncalibrated):\')\n    display = pd.DataFrame(glyph_rows)\n    display[\'input\'] = display[\'input\'].map(lambda p: str(p).replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1])\n    print(display.round(2).to_string(index=False))\n    if show_stroke_table:\n        print(\'\\nPer-stroke scores:\')\n        print(pd.DataFrame(stroke_rows)[[\'sample\', \'stroke\', \'missing\', \'split\', \'junction\', \'stroke_score\']]\n              .to_string(index=False, float_format=lambda x: f\'{x:.4g}\'))\n    print(\'Image : score gallery:\', gallery)\n    print(\'Saved:\', destination)\n    return output_report\n\n\ndef export_indicator_debug(structure_report, indicator_report, destination=None):\n    """Package only evaluated snapshots and reports, never a checkpoint/dataset."""\n    import copy\n    import zipfile\n    source_path = None\n    if not isinstance(structure_report, dict):\n        source_path = Path(structure_report)\n        structure_report = json.loads(source_path.read_text(encoding=\'utf-8-sig\'))\n    report = copy.deepcopy(structure_report)\n    scores = indicator_report[\'results\']\n    if len(report[\'results\']) != len(scores):\n        raise ValueError(\'Structure and indicator reports must describe the same evaluated samples\')\n    path = Path(destination or Path(indicator_report[\'save_dir\'])/\'indicator_debug.zip\')\n    path.parent.mkdir(parents=True, exist_ok=True)\n    summary, missing = [], []\n    with zipfile.ZipFile(path, \'w\', compression=zipfile.ZIP_DEFLATED) as archive:\n        for number, (sample, score) in enumerate(zip(report[\'results\'], scores), 1):\n            found = None\n            for key in (\'npz\', \'input_npz\'):\n                if not sample.get(key):\n                    continue\n                candidates = [Path(sample[key])]\n                if source_path and not candidates[0].is_absolute():\n                    candidates.append(source_path.parent/candidates[0])\n                found = next((p for p in candidates if p.is_file()), None)\n                if found:\n                    break\n            if found:\n                name = f\'arrays/sample_{number:04d}.npz\'\n                archive.write(found, name)\n                sample[\'npz\'] = name\n                sample.pop(\'input_npz\', None)\n            else:\n                missing.append(number)\n            caps = {key: score[key+\'_score_cap\'] for key in (\'missing\', \'extra\', \'junction\', \'split\')}\n            limiting = min(caps, key=caps.get)\n            if score[\'structural_score\'] < min(caps.values())-1e-5:\n                limiting = \'combined\'\n            summary.append({\'sample\': number, \'input\': score.get(\'input\'),\n                \'score\': score[\'structural_score\'], \'limiting_term\': limiting,\n                \'score_caps\': caps, \'strokes\': score[\'strokes\']})\n        archive.writestr(\'structure_report.json\', json.dumps(report, ensure_ascii=False, indent=2))\n        archive.writestr(\'indicator_report.json\', json.dumps(indicator_report, ensure_ascii=False, indent=2))\n        archive.writestr(\'diagnosis.json\', json.dumps({\'samples\': summary, \'missing_arrays\': missing,\n            \'purpose\': \'Diagnose false penalties; not a validated quality assessment.\'}, ensure_ascii=False, indent=2))\n    print(\'Diagnostic ZIP:\', path)\n    if missing:\n        print(\'Snapshots unavailable for samples:\', missing)\n    return str(path)\n', _score_module.__file__, "exec"), _score_module.__dict__)


In [ ]:
from _structural_indicator_runtime import IndicatorConfig, score_structure_report
from pathlib import Path

if MODE == "csv":
    import random
    import pandas as pd
    from stroke_model.convert_reference_csv import existing_path
    from stroke_model.infer_reference import inference_columns, select_input_files
    from stroke_model.stroke_structure import StructureConfig
    from stroke_model.score_stroke_structure import score_structure_csv
    csv_path = existing_path(CSV_PATH)
    frame = pd.read_csv(csv_path)
    columns = inference_columns(frame)
    if INPUT_FILES is not None:
        selected_rows = select_input_files(frame, columns['image'], INPUT_FILES)
    else:
        if SAMPLE_COUNT is not None and (not isinstance(SAMPLE_COUNT, int) or SAMPLE_COUNT < 1):
            raise ValueError("SAMPLE_COUNT는 양의 정수 또는 None이어야 합니다.")
        candidates = list(range(1, len(frame)+1))
        selected_rows = (candidates if SAMPLE_COUNT is None else
                         sorted(random.Random(RANDOM_SEED).sample(candidates, min(SAMPLE_COUNT, len(candidates)))))
    if not selected_rows:
        raise ValueError("선택한 이미지가 없습니다. CSV와 INPUT_FILES를 확인하세요.")
    print("평가할 테스트 이미지:")
    print(pd.DataFrame({'csv_row': selected_rows,
        'input': [frame.iloc[i-1][columns['image']] for i in selected_rows]}).to_string(index=False))
    # CSV 모드는 이전 structure_results와 무관하게 이번에 선택한 사진을 추론합니다.
    structure_results = score_structure_csv(
        CHECKPOINT, csv_path, sample_rows=selected_rows, dataset_root=DATASET_ROOT,
        config=StructureConfig(**STRUCTURE_CONFIG), geometry="reference", show=False,
    )
    indicator_source = structure_results
elif MODE == "values":
    indicator_source = {"results": [{"input": "manual_values", "strokes": VALUES, "extra": EXTRA_VALUE}]}
elif MODE == "auto":
    indicator_source = globals().get("structure_results")
    if indicator_source is None:
        if not REPORT_PATH:
            raise ValueError("구조 진단을 먼저 실행하거나 REPORT_PATH에 structure_report.json 경로를 넣으세요.")
        if str(REPORT_PATH).startswith("/content/drive/"):
            from google.colab import drive
            drive.mount("/content/drive")
        indicator_source = Path(REPORT_PATH)
else:
    raise ValueError('MODE는 "csv", "auto", "values"여야 합니다.')

indicator_results = score_structure_report(
    indicator_source, config=IndicatorConfig(**SCORE_CONFIG), save_dir=SAVE_DIR,
    show_images=SHOW_IMAGES, images_per_row=IMAGES_PER_ROW,
    images_per_page=IMAGES_PER_PAGE, show_stroke_table=SHOW_STROKE_TABLE,
)
if EXPORT_DEBUG and MODE != "values":
    from _structural_indicator_runtime import export_indicator_debug
    debug_zip = export_indicator_debug(indicator_source, indicator_results)
    try:
        from google.colab import files
        import ipywidgets as widgets
        from IPython.display import display
        _download = widgets.Button(description="진단 ZIP 다운로드")
        _download.on_click(lambda _: files.download(debug_zip))
        display(_download)
    except ImportError:
        pass
